# Data Retrieval Reddit

| Field | Details |
|---|---|
| **Time window** | 5 Jul 2024 – 4 Nov 2024 |
| **Source** | Reddit — JSONL files (Pushshift / Academic Data Access) |
| **Collection** | Pre-downloaded before Reddit API shutdown by external contributor; not collected by this team |
| **Subreddits** | r/conservative · r/democrats · r/republican · r/worldnews · r/politics · r/liberal · r/trump |
| **Method** | Merge all per-subreddit JSONL files; filter inline to date window and election keywords (same keyword set as Bluesky) |
| **Keywords** | trump · harris · kamala · biden · vance · walz · election · vote · ballot · debate · maga · gop · dnc · rnc · swing state · battleground · project2025 · … |
| **Saved to** | `Data/1_Bronze/Reddit/reddit_comments_raw.parquet` · `Data/1_Bronze/Reddit/reddit_posts_raw.parquet` |

### reddit_comments_raw.parquet

| Column | Type | Description |
|---|---|---|
| `subreddit_source` | str | Subreddit name (conservative / democrats / republican / worldnews / politics / liberal / trump) |
| `datetime` | datetime (UTC) | Comment timestamp, derived from `created_utc` |
| `id` | str | Reddit comment ID |
| `author` | str | Reddit username of the commenter |
| `body` | str | Comment text |
| `score` | int | Net upvote score |
| `created_utc` | int | Unix timestamp of creation |
| `link_id` | str | Root post ID (`t3_XXXXX`) — links comment back to its parent post |
| `parent_id` | str | Direct parent ID (`t1_XXXXX` = parent comment · `t3_XXXXX` = direct reply to post) |

### reddit_posts_raw.parquet

| Column | Type | Description |
|---|---|---|
| `subreddit_source` | str | Subreddit name |
| `datetime` | datetime (UTC) | Post timestamp, derived from `created_utc` |
| `id` | str | Reddit post ID |
| `author` | str | Reddit username of the poster |
| `title` | str | Post title |
| `selftext` | str | Post body text (empty for link posts) |
| `score` | int | Net upvote score |
| `num_comments` | int | Total number of comments on the post |
| `upvote_ratio` | float | Fraction of upvotes out of total votes (0–1) |
| `created_utc` | int | Unix timestamp of creation |

In [ ]:
import json
from pathlib import Path
import pandas as pd

BRONZE_DIR = Path('../Data/1_Bronze/Reddit')

SUBREDDITS = ['conservative', 'democrats', 'republican', 'worldnews', 'politics', 'liberal', 'trump']

# Time window
DATE_START = pd.Timestamp('2024-07-05', tz='UTC')
DATE_END   = pd.Timestamp('2024-11-04 23:59:59', tz='UTC')

# Election keywords — mirrors the Bluesky hashtag list, converted to plain text
# A comment or post must mention at least one of these to be kept
ELECTION_KEYWORDS = [
    # Candidates & running mates
    'trump', 'donald trump', 'harris', 'kamala', 'kamala harris',
    'biden', 'joe biden', 'vance', 'jd vance', 'walz', 'tim walz',

    # Election process
    'election', 'election2024', 'us election', 'electionday',
    'vote', 'vote2024', 'voting', 'voter', 'voterregistration',
    'ballot', 'earlyvoting', 'mail-in voting', 'mailinvoting',
    'poll', 'polls', 'electoral', 'swing state', 'battleground',
    'electionintegrity', 'america decides', 'decision2024',

    # Campaign & parties
    'campaign', 'debate', 'presidential debate', 'vp debate',
    'trumpdebate', 'trumpharrisdebate', 'debatenight',
    'president', 'presidential',
    'democrat', 'democratic', 'democrats',
    'republican', 'republicans',
    'maga', 'maga2024', 'america first', 'americafirst',
    'gop', 'dnc', 'rnc',
    'project2025',

    # Conventions & events
    'convention', 'rnc2024', 'dnc2024', 'republican convention',
    'democratic convention',
    'trump rally', 'trumprally',
    'inauguration', 'primary',

    # Biden exit
    'biden drops out', 'bidendropsout', 'biden out', 'biden withdraws',

    # Slogans
    'vote blue', 'voteblue', 'vote trump', 'votetrump',
    'vote harris', 'voteharris', 'vote kamala',
    'we are not going back', 'harriswalz', 'trumpvance',
]

In [ ]:
# Load all comment JSONL files, filter to election window + election keywords, then merge.
# Each comment retains link_id (-> root post) and parent_id (-> direct parent).
comment_frames = []

for sub in SUBREDDITS:
    path = BRONZE_DIR / f'r_{sub}_comments.jsonl'
    if not path.exists():
        print(f'MISSING: {path.name}')
        continue
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            # Date filter
            utc = obj.get('created_utc')
            if utc is None:
                continue
            dt = pd.Timestamp(int(utc), unit='s', tz='UTC')
            if not (DATE_START <= dt <= DATE_END):
                continue
            # Keyword filter on body
            body = obj.get('body', '') or ''
            if not any(kw in body.lower() for kw in ELECTION_KEYWORDS):
                continue
            obj['subreddit_source'] = sub
            obj['datetime'] = dt
            rows.append(obj)
    df_sub = pd.DataFrame(rows)
    comment_frames.append(df_sub)
    print(f'r/{sub}: {len(df_sub):,} election comments kept')

df_comments = pd.concat(comment_frames, ignore_index=True)
print(f'\nTotal comments: {df_comments.shape[0]:,} rows x {df_comments.shape[1]} columns')

In [ ]:
# Load all post JSONL files, filter to election window + election keywords, then merge.
# Keyword is checked on title and selftext combined.
post_frames = []

for sub in SUBREDDITS:
    path = BRONZE_DIR / f'r_{sub}_posts.jsonl'
    if not path.exists():
        print(f'MISSING: {path.name}')
        continue
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            # Date filter
            utc = obj.get('created_utc')
            if utc is None:
                continue
            dt = pd.Timestamp(int(utc), unit='s', tz='UTC')
            if not (DATE_START <= dt <= DATE_END):
                continue
            # Keyword filter on title + selftext
            text = (obj.get('title', '') or '') + ' ' + (obj.get('selftext', '') or '')
            if not any(kw in text.lower() for kw in ELECTION_KEYWORDS):
                continue
            obj['subreddit_source'] = sub
            obj['datetime'] = dt
            rows.append(obj)
    df_sub = pd.DataFrame(rows)
    post_frames.append(df_sub)
    print(f'r/{sub}: {len(df_sub):,} election posts kept')

df_posts = pd.concat(post_frames, ignore_index=True)
print(f'\nTotal posts: {df_posts.shape[0]:,} rows x {df_posts.shape[1]} columns')

In [ ]:
# Quick volume check per subreddit and verify key linkage columns are present
print('=== Comments ===')
print(df_comments['subreddit_source'].value_counts())
print(f'\nlink_id present: {"link_id" in df_comments.columns}')
print(f'parent_id present: {"parent_id" in df_comments.columns}')

print('\n=== Posts ===')
print(df_posts['subreddit_source'].value_counts())

In [ ]:
# Save raw combined files to Bronze
out_comments = BRONZE_DIR / 'reddit_comments_raw.parquet'
out_posts    = BRONZE_DIR / 'reddit_posts_raw.parquet'

df_comments.to_parquet(out_comments, index=False)
df_posts.to_parquet(out_posts, index=False)

print(f'Saved:')
print(f'  {out_comments}  ({df_comments.shape[0]:,} rows x {df_comments.shape[1]} columns)')
print(f'  {out_posts}  ({df_posts.shape[0]:,} rows x {df_posts.shape[1]} columns)')